# 06 — Night dataset evaluation (Colab)

**Ночной тест:** в Google Drive `MyDrive/Colab Notebooks/prepared/` должны лежать:
- `night_drones_vs_planes.yolov7pytorch.zip` — YOLO (Roboflow / Ultralytics `val`)
- `night_drones_vs_planes.coco.zip` — COCO (структура для опционального Faster R-CNN)

Распаковка в **`/content/night_eval/`** (локальный диск сессии Colab).  
Если в YOLO-архиве только **`train/`** (без val/test), **CELL 3** сама делит файлы **70 % / 15 % / 15 %** (train/val/test, `seed=42`) и обновляет `data.yaml`.

Артефакты: **`results/night_eval/`** (не перезаписывают вывод `04_evaluation.ipynb`).

**Согласование классов:** веса обучены на **4 класса**. Если в архиве **только 2 класса** (plane + drone), CELL 4 **автоматически** переписывает YOLO-лейблы: plane→`AIRPLANE` (1), drone→`DRONE` (0), выставляет `nc: 4` и имена как в основном проекте. Классы HELICOPTER/BIRD в разметке ночного набора просто отсутствуют.


In [ ]:
# ── CELL 1: Install & GPU ─────────────────────────────────────────────────────
!uv pip install ultralytics pycocotools seaborn scikit-learn pyyaml
import ultralytics

ultralytics.checks()
!nvidia-smi
import torch

assert torch.cuda.is_available(), "No GPU!"
print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ── CELL 2: Mount Drive ───────────────────────────────────────────────────────
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# ── CELL 3: Unzip night datasets + при необходимости train→train/val/test ───
import random
import shutil
import zipfile
from pathlib import Path

import yaml

DRIVE_PREPARED = Path("/content/drive/MyDrive/Colab Notebooks/prepared")
NIGHT_ROOT = Path("/content/night_eval")
NIGHT_YOLO_DIR = NIGHT_ROOT / "yolo_unpack"
NIGHT_COCO_DIR = NIGHT_ROOT / "coco_unpack"

YOLO_ZIP_NAME = "night_drones_vs_planes.yolov7pytorch.zip"
COCO_ZIP_NAME = "night_drones_vs_planes.coco.zip"

# Доли после распаковки, если в архиве только один split `train`
NIGHT_SPLIT_SEED = 42
NIGHT_TRAIN_FRAC = 0.70
NIGHT_VAL_FRAC = 0.15
NIGHT_TEST_FRAC = 0.15

_IMG_EXT_UNZIP = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def unzip_to(zip_path: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest)
    print(f"OK: {zip_path.name} → {dest}")


def _dir_has_any_images(p: Path) -> bool:
    if not p.is_dir():
        return False
    return any(x.is_file() and x.suffix.lower() in _IMG_EXT_UNZIP for x in p.iterdir())


def _find_yolo_dataset_root(search: Path) -> Path | None:
    """Корень с train/images (родитель папки train); сначала мелковложенный путь."""
    for timg in sorted(search.rglob("train/images"), key=lambda x: len(x.parts)):
        if timg.is_dir() and _dir_has_any_images(timg):
            return timg.parent.parent.resolve()
    return None


def split_single_train_to_val_test(ds_root: Path) -> bool:
    """Если есть только train с картинками, а val/test пустые — делим 70/15/15 и двигаем файлы."""
    tr_img = ds_root / "train" / "images"
    tr_lbl = ds_root / "train" / "labels"
    if not tr_img.is_dir() or not _dir_has_any_images(tr_img):
        return False
    val_ok = _dir_has_any_images(ds_root / "val" / "images")
    te_ok = _dir_has_any_images(ds_root / "test" / "images")
    if val_ok and te_ok:
        print("Разбиение не нужно: уже есть непустые val/ и test/.")
        return False

    stems = sorted(
        {
            p.stem
            for p in tr_img.iterdir()
            if p.is_file() and p.suffix.lower() in _IMG_EXT_UNZIP
        }
    )
    n = len(stems)
    if n < 6:
        print(f"Мало изображений ({n}) — разбиение train/val/test пропущено.")
        return False

    rng = random.Random(NIGHT_SPLIT_SEED)
    rng.shuffle(stems)
    i1 = int(round(n * NIGHT_TRAIN_FRAC))
    i2 = i1 + int(round(n * NIGHT_VAL_FRAC))
    i1 = max(1, min(i1, n - 2))
    i2 = max(i1 + 1, min(i2, n - 1))
    st_train, st_val, st_test = stems[:i1], stems[i1:i2], stems[i2:]
    if not st_val:
        st_val = [st_train.pop()]
    if not st_test:
        st_test = [st_train.pop()]

    for sub in ("val", "test"):
        (ds_root / sub / "images").mkdir(parents=True, exist_ok=True)
        (ds_root / sub / "labels").mkdir(parents=True, exist_ok=True)

    def _move_stem(stem: str, split: str) -> None:
        dst_img = ds_root / split / "images"
        dst_lbl = ds_root / split / "labels"
        for p in tr_img.glob(stem + ".*"):
            if p.suffix.lower() in _IMG_EXT_UNZIP and p.is_file():
                shutil.move(str(p), str(dst_img / p.name))
                break
        lp = tr_lbl / f"{stem}.txt"
        if lp.is_file():
            shutil.move(str(lp), str(dst_lbl / f"{stem}.txt"))

    for s in st_val:
        _move_stem(s, "val")
    for s in st_test:
        _move_stem(s, "test")

    _guard = ds_root / ".night_thesis_remap_done"
    if _guard.is_file():
        _guard.unlink()
        print("Удалён маркер", _guard, "— после разбиения нужен свежий remapping в CELL 4.")

    _yl = sorted(ds_root.rglob("data.yaml"))
    ypath = _yl[0] if _yl else ds_root / "data.yaml"
    yd = {}
    if ypath.is_file():
        with open(ypath, encoding="utf-8") as f:
            yd = yaml.safe_load(f) or {}
    yd["path"] = str(ds_root.resolve())
    yd["train"] = "train/images"
    yd["val"] = "val/images"
    yd["test"] = "test/images"
    with open(ypath, "w", encoding="utf-8") as f:
        yaml.dump(yd, f, allow_unicode=True, default_flow_style=False, sort_keys=False)
    print(f"Обновлён data.yaml: {ypath}")
    print(
        f"Разбиение: train={len(st_train)} | val={len(st_val)} | test={len(st_test)} | seed={NIGHT_SPLIT_SEED} "
        f"({NIGHT_TRAIN_FRAC:.0%}/{NIGHT_VAL_FRAC:.0%}/{NIGHT_TEST_FRAC:.0%})"
    )
    return True


for zname, dest in ((YOLO_ZIP_NAME, NIGHT_YOLO_DIR), (COCO_ZIP_NAME, NIGHT_COCO_DIR)):
    zp = DRIVE_PREPARED / zname
    if not zp.exists():
        raise FileNotFoundError(f"Нет файла: {zp}. Положите zip в prepared/ на Drive.")
    unzip_to(zp, dest)

_ds = _find_yolo_dataset_root(NIGHT_YOLO_DIR)
if _ds is not None:
    print("Корень YOLO-датасета:", _ds)
    split_single_train_to_val_test(_ds)
else:
    print("Не найден train/images под", NIGHT_YOLO_DIR)

print("Корень ночных данных:", NIGHT_ROOT)


In [ ]:
# ── CELL 4: Найти data.yaml, патч path / val для Ultralytics ─────────────────
from pathlib import Path

import yaml

# Первый data.yaml внутри YOLO-распаковки
_cands = sorted(NIGHT_YOLO_DIR.rglob("data.yaml"))
if not _cands:
    raise FileNotFoundError(f"Нет data.yaml под {NIGHT_YOLO_DIR}")
YAML_NIGHT = _cands[0]
print("Исходный YOLO data.yaml:", YAML_NIGHT)

with open(YAML_NIGHT, encoding="utf-8") as f:
    _raw = yaml.safe_load(f)
if _raw is None:
    _raw = {}

if "valid" in _raw and "val" not in _raw:
    _raw["val"] = _raw.pop("valid")

_yp = YAML_NIGHT.parent.resolve()
_IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def _dir_has_images(p: Path) -> bool:
    if not p.is_dir():
        return False
    for x in p.iterdir():
        if x.is_file() and x.suffix.lower() in _IMG_EXT:
            return True
    return False


def _rel_split_with_images(ds: Path, split: str) -> str | None:
    """Относительный путь от ds к папке с изображениями: train/images, valid, test/images и т.д."""
    for rel in (f"{split}/images", split):
        p = ds / rel
        if _dir_has_images(p):
            return rel.replace("\\", "/")
    return None


def _find_split(ds: Path, *aliases: str) -> str | None:
    for a in aliases:
        r = _rel_split_with_images(ds, a)
        if r:
            return r
    lower = {a.lower() for a in aliases}
    for sub in ds.iterdir():
        if sub.is_dir() and sub.name.lower() in lower:
            r = _rel_split_with_images(ds, sub.name)
            if r:
                return r
    return None


def _train_candidate_dirs(search_root: Path) -> list[Path]:
    out: list[Path] = []
    for td in sorted(search_root.rglob("train"), key=lambda p: len(p.parts)):
        if not td.is_dir():
            continue
        if (td / "images").is_dir() and _dir_has_images(td / "images"):
            out.append(td)
        elif _dir_has_images(td):
            out.append(td)
    return out


# Выбираем корень датасета: родитель train с картинками; при нескольких — где есть val/test
_ds_root = _yp
_best_sc = -1
for _td in _train_candidate_dirs(_yp):
    _cand = _td.parent
    _ct = _find_split(_cand, "train")
    _cv = _find_split(_cand, "valid", "val", "validation")
    _cte = _find_split(_cand, "test")
    _sc = (3 if _cv else 0) + (2 if _ct else 0) + (1 if _cte else 0)
    if _sc > _best_sc:
        _best_sc = _sc
        _ds_root = _cand
if _best_sc <= 0:
    _ds_root = _yp
_ds_root = _ds_root.resolve()
print("Корень датасета (path в yaml):", _ds_root)
print("Содержимое корня:", [p.name for p in _ds_root.iterdir() if p.is_dir()])


_tr = _find_split(_ds_root, "train")
if _tr is None:
    _tr = "train/images" if (_ds_root / "train" / "images").is_dir() else str(_raw.get("train", "train/images"))

_va = _find_split(_ds_root, "valid", "val", "validation")
_te = _find_split(_ds_root, "test")

if _va is None and _te is not None:
    print("Нет valid/val — для ключа val в yaml используем test (типично для Roboflow export).")
    _va = _te
if _va is None and _tr is not None:
    print("ВНИМАНИЕ: нет val/test — подставляем train в val (только для прохождения val()).")
    _va = _tr

_fix = {"path": str(_ds_root), "train": _tr, "val": _va}
if _te is not None and _te != _va:
    _fix["test"] = _te

for _check_k, _check_v in (("train", _tr), ("val", _va)):
    _cp = _ds_root / _check_v
    _nimg = sum(1 for x in _cp.iterdir() if x.is_file() and x.suffix.lower() in _IMG_EXT) if _cp.is_dir() else 0
    print(f"  {_check_k} → {_check_v} | exists={_cp.is_dir()} | n_images={_nimg}")
for k in ("nc", "names", "download"):
    if k in _raw:
        _fix[k] = _raw[k]

_names = _fix.get("names")
if isinstance(_names, dict):
    _fix["names"] = [_names[i] for i in sorted(_names.keys())]
elif not isinstance(_names, list):
    _fix["names"] = ["DRONE", "AIRPLANE", "HELICOPTER", "BIRD"]

if "nc" not in _fix:
    _fix["nc"] = len(_fix["names"])

THESIS_NAMES = ["DRONE", "AIRPLANE", "HELICOPTER", "BIRD"]
NIGHT_CLASS_REMAP_APPLIED = False


def _alias_to_thesis_id(name: str) -> int:
    """Сопоставление имён Roboflow → индекс в 4-классовой схеме проекта."""
    s = str(name).strip().lower().replace("_", " ")
    if s in ("drone", "uav", "quadcopter", "multirotor"):
        return 0
    if s in ("plane", "airplane", "aeroplane", "aircraft", "fixed wing", "fixed-wing"):
        return 1
    raise ValueError(
        f"Класс {name!r} не распознан. Отредактируйте _alias_to_thesis_id в CELL 4."
    )


def _images_dir_to_labels_dir(images_dir: Path) -> Path:
    parts = list(images_dir.parts)
    if "images" in parts:
        i = parts.index("images")
        parts[i] = "labels"
        return Path(*parts)
    return images_dir.parent / "labels"


_old_nc = int(_fix["nc"])
_old_names = list(_fix["names"])

_remap_guard = _ds_root / ".night_thesis_remap_done"
if _old_nc == 2 and len(_old_names) == 2 and _remap_guard.exists():
    print(
        "Remapping лейблов уже выполнялся ранее — пропуск перезаписи .txt (защита от двойного прогона). "
        f"Чтобы сделать заново: удалите {_remap_guard} и заново распакуйте YOLO-zip (CELL 3)."
    )
    _fix["nc"] = 4
    _fix["names"] = THESIS_NAMES
    NIGHT_CLASS_REMAP_APPLIED = True
elif _old_nc == 2 and len(_old_names) == 2:
    print("2 класса в датасете → remapping в nc=4 (DRONE=0, AIRPLANE=1; HELI/BIRD без GT).")
    _oid_to_thesis = {i: _alias_to_thesis_id(_old_names[i]) for i in range(2)}
    print("  старый id → thesis id:", _oid_to_thesis, "| имена:", _old_names)
    _n_txt = 0
    _seen_lbl_root: set = set()
    _seen_txt: set = set()
    for _sk in ("train", "val", "test"):
        _rel = _fix.get(_sk)
        if not _rel:
            continue
        _img_dir = (_ds_root / _rel).resolve()
        _lbl = _images_dir_to_labels_dir(_img_dir)
        if not _lbl.is_dir():
            _lbl2 = _ds_root / Path(_rel).parent / "labels"
            _lbl = _lbl2 if _lbl2.is_dir() else _lbl
        if not _lbl.is_dir():
            print(f"  пропуск split {_sk}: нет папки labels рядом с {_img_dir}")
            continue
        _lr = _lbl.resolve()
        if _lr in _seen_lbl_root:
            print(
                f"  split {_sk}: тот же каталог labels, что уже обработан ({_lr}) — без повторного remapping."
            )
            continue
        _seen_lbl_root.add(_lr)
        for _tf in _lbl.rglob("*.txt"):
            _tf_r = _tf.resolve()
            if _tf_r in _seen_txt:
                continue
            _seen_txt.add(_tf_r)
            _lines_out = []
            for _line in _tf.read_text(encoding="utf-8", errors="ignore").splitlines():
                _line = _line.strip()
                if not _line:
                    continue
                _parts = _line.split()
                _oi = int(float(_parts[0]))
                if _oi not in _oid_to_thesis:
                    raise ValueError(f"Неизвестный class id {_oi} в {_tf}")
                _parts[0] = str(_oid_to_thesis[_oi])
                _lines_out.append(" ".join(_parts))
            _tf.write_text("\n".join(_lines_out) + ("\n" if _lines_out else ""), encoding="utf-8")
            _n_txt += 1
    print(f"  перезаписано label-файлов: {_n_txt}")
    _fix["nc"] = 4
    _fix["names"] = THESIS_NAMES
    NIGHT_CLASS_REMAP_APPLIED = True
    try:
        _remap_guard.write_text("nc4\n", encoding="utf-8")
    except OSError as exc:
        print(f"  не удалось записать маркер {_remap_guard}: {exc}")

YAML_NIGHT_FIXED = NIGHT_ROOT / "night_data.yaml"
with open(YAML_NIGHT_FIXED, "w", encoding="utf-8") as f:
    f.write("# Patched for night eval (Colab)\n")
    yaml.dump(_fix, f, allow_unicode=True, default_flow_style=False, sort_keys=False)

print("Сохранён:", YAML_NIGHT_FIXED)
print("nc=", _fix["nc"], "names=", _fix["names"])

CLASS_NAMES = list(_fix["names"])
NC_NIGHT = int(_fix["nc"])
YAML_PATH = YAML_NIGHT_FIXED


In [ ]:
# ── CELL 5: Веса, папка результатов, split для val() ─────────────────────────
import json
from pathlib import Path

import yaml

DRIVE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks")
WEIGHTS_DIR = DRIVE_ROOT / "weights"
RESULTS_DIR = DRIVE_ROOT / "results" / "night_eval"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

YOLO_WEIGHT_FILES = [
    ("YOLOv12n", "yolo12n_drone_best.pt"),
    ("YOLOv12s", "yolo12s_drone_best.pt"),
    ("YOLO26n", "yolo26n_drone_best.pt"),
    ("YOLO26s", "yolo26s_drone_best.pt"),
]
_JSON_KEY = {"YOLOv12n": "yolo12n", "YOLOv12s": "yolo12s", "YOLO26n": "yolo26n", "YOLO26s": "yolo26s"}

MODEL_COLORS = {
    "YOLOv12n": "#e74c3c",
    "YOLOv12s": "#3498db",
    "YOLO26n": "#8e44ad",
    "YOLO26s": "#16a085",
    "Faster R-CNN": "#27ae60",
}

PALETTE = ["#e74c3c", "#3498db", "#2ecc71", "#f1c40f", "#9b59b6", "#1abc9c"]

_IMG_EXT_SPLIT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def _yolo_labels_dir_for_images(img_dir: Path) -> Path | None:
    """.../split/images → .../split/labels (YOLO layout)."""
    if img_dir.name.lower() != "images":
        return None
    ld = img_dir.parent / "labels"
    return ld if ld.is_dir() else None


def _split_image_label_stats(root: Path, rel: str) -> tuple[int, int]:
    """Число (stem) совпадений картинка↔txt и всего картинок."""
    img_dir = root / rel
    if not img_dir.is_dir():
        return 0, 0
    stems_img = {
        p.stem for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in _IMG_EXT_SPLIT
    }
    nimg = len(stems_img)
    if nimg == 0:
        return 0, 0
    ld = _yolo_labels_dir_for_images(img_dir)
    if ld is None:
        return 0, nimg
    stems_lbl = {p.stem for p in ld.rglob("*.txt")}
    return len(stems_img & stems_lbl), nimg


def _pick_split(yaml_path: Path) -> str:
    """Сначала val (валидация), не test: иначе часто берётся пустой/безлейбловый test → mAP≈0."""
    with open(yaml_path, encoding="utf-8") as f:
        yd = yaml.safe_load(f)
    if not isinstance(yd, dict):
        return "val"
    root = yd.get("path")
    if root is None or str(root).strip() in ("", "."):
        root = yaml_path.parent
    else:
        root = Path(root)
        if not root.is_absolute():
            root = (yaml_path.parent / root).resolve()
        else:
            root = root.resolve()
    best_sp, best_common = None, -1
    for sp in ("val", "test"):
        rel = yd.get(sp)
        if not rel:
            continue
        common, nimg = _split_image_label_stats(root, str(rel).replace("\\", "/"))
        ok = nimg > 0 and common >= max(1, int(0.5 * nimg))
        print(f"  split {sp!r}: rel={rel!r} | images={nimg} | paired_labels={common} | ok={ok}")
        if ok and common > best_common:
            best_common = common
            best_sp = sp
    if best_sp is not None:
        return best_sp
    for sp in ("val", "test"):
        rel = yd.get(sp)
        if not rel:
            continue
        p = root / rel
        if p.is_dir() and any(p.iterdir()):
            print(f"  fallback: {sp} (мало парных лейблов — проверьте папки images/labels).")
            return sp
    return "val"


SPLIT_NIGHT = _pick_split(YAML_PATH)
print("split для val():", SPLIT_NIGHT)
print("RESULTS_DIR:", RESULTS_DIR)

if NC_NIGHT == 4 and globals().get("NIGHT_CLASS_REMAP_APPLIED"):
    print("Классы согласованы с весами: 2→4 remapping из CELL 4 (drone/plane → DRONE/AIRPLANE).")
elif NC_NIGHT != 4:
    print()
    print("!" * 60)
    print("ВНИМАНИЕ: nc в ночном датасете =", NC_NIGHT, "— веса проекта на 4 класса.")
    print("Если val() упадёт: remapping для 2 классов в CELL 4 или приведите nc/names вручную.")
    print("!" * 60)


## 1. YOLO: val() на ночном датасете → `night_yolo_metrics.json`


In [ ]:
# ── CELL 6: Запуск val() для каждого доступного .pt, сохранение метрик ────────
import json
from pathlib import Path

import numpy as np
import torch
import yaml
from ultralytics import YOLO

with open(YAML_PATH, encoding="utf-8") as _yf:
    _yev = yaml.safe_load(_yf)
_root_ev = Path(_yev.get("path") or YAML_PATH.parent).resolve()
_rel_sp = _yev.get(SPLIT_NIGHT) or _yev.get("val")
print("Проверка split для val():", SPLIT_NIGHT, "→", _root_ev / _rel_sp if _rel_sp else "?")

night_yolo_metrics: dict = {}
night_per_class: dict = {}
_val_errors: list[dict[str, str]] = []
_skipped_weights: list[str] = []

for label, fname in YOLO_WEIGHT_FILES:
    key = _JSON_KEY[label]
    wp = WEIGHTS_DIR / fname
    if not wp.exists():
        print(f"Skip {label}: нет {wp}")
        _skipped_weights.append(f"{label}: {fname}")
        continue
    print(f"\n>>> {label} — val на ночном split={SPLIT_NIGHT}")
    try:
        model = YOLO(str(wp))
        val = model.val(
            data=str(YAML_PATH),
            split=SPLIT_NIGHT,
            device=0,
            imgsz=640,
            batch=16,
            verbose=False,
            plots=False,
        )
    except Exception as exc:
        print(f"Ошибка val() для {label}: {exc}")
        _val_errors.append({"model": label, "error": str(exc)})
        continue
    try:
        mp = float(val.box.mp)
        mr = float(val.box.mr)
    except AttributeError:
        mr1 = val.mean_results()
        mp, mr = float(mr1[0]), float(mr1[1])
    night_yolo_metrics[key] = {
        "mAP50": round(float(val.box.map50), 4),
        "mAP50_95": round(float(val.box.map), 4),
        "precision": round(mp, 4),
        "recall": round(mr, 4),
        "per_class_mAP50": [round(float(x), 4) for x in val.box.ap50.tolist()],
        "fps": None,
        "size_mb": round(wp.stat().st_size / 1e6, 1),
        "dataset": "night_drones_vs_planes",
        "split": SPLIT_NIGHT,
    }
    night_per_class[label] = dict(zip(CLASS_NAMES, val.box.ap50.tolist()))
    print(f"  mAP50={night_yolo_metrics[key]['mAP50']}, mAP50-95={night_yolo_metrics[key]['mAP50_95']}")

night_yolo_metrics["class_names"] = CLASS_NAMES
night_yolo_metrics["_meta"] = {
    "yaml": str(YAML_PATH),
    "split": SPLIT_NIGHT,
    "ok_models": [k for k in _JSON_KEY.values() if k in night_yolo_metrics and isinstance(night_yolo_metrics[k], dict)],
    "skipped_weights": _skipped_weights,
    "val_errors": _val_errors,
}
out_json = RESULTS_DIR / "night_yolo_metrics.json"
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(night_yolo_metrics, f, indent=2, ensure_ascii=False)
print(f"\nСохранено: {out_json}")
if not night_yolo_metrics["_meta"]["ok_models"]:
    print(
        "\n!!! Ни одна модель не прошла val() — в JSON только class_names и _meta. "
        "Проверьте пути в CELL 4, веса в WEIGHTS_DIR и поле _meta['val_errors']."
    )


## 1b. Faster R-CNN на ночном COCO + сравнение с **YOLOv12s**

Нужны веса в `weights/`: **`faster_rcnn_best.pth`**, **`yolo12s_drone_best.pt`**.  
Разметка — из **`night_drones_vs_planes.coco.zip`**: ищутся `instances_test.json` / `instances_val.json` / `instances_train.json` / `_annotations.coco.json` (Roboflow часто кладёт train).  
Метрики R-CNN: pycocotools (как в `03_faster_rcnn_train.ipynb`); **recall** = COCO AR@100 (не как у YOLO).

Результаты: `night_rcnn_metrics.json`, `night_compare_yolo12s_rcnn.csv`.

In [ ]:
# ── CELL 6b: Faster R-CNN night COCO + сравнение с YOLOv12s ─────────────────────
import json
import math
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as tvf
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from tqdm import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

RCNN_WEIGHTS = WEIGHTS_DIR / "faster_rcnn_best.pth"
YOLO12S_NAME = "yolo12s_drone_best.pt"

DEVICE_RCNN = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES_RCNN = 5  # 4 класса + фон


def _parse_coco_bbox(ann: dict):
    bb = ann.get("bbox")
    if not bb or len(bb) < 4:
        return None
    try:
        x, y, w, h = map(float, bb[:4])
    except (TypeError, ValueError):
        return None
    if w <= 0 or h <= 0 or not all(math.isfinite(v) for v in (x, y, w, h)):
        return None
    return x, y, w, h


def _clip_xyxy_to_image(boxes, labels, height: int, width: int):
    out_b, out_l = [], []
    w, h = float(width), float(height)
    for box, lab in zip(boxes, labels):
        x1, y1, x2, y2 = (float(v) for v in box)
        x1, x2 = max(0.0, min(x1, w)), max(0.0, min(x2, w))
        y1, y2 = max(0.0, min(y1, h)), max(0.0, min(y2, h))
        if x2 <= x1 or y2 <= y1:
            continue
        out_b.append([x1, y1, x2, y2])
        out_l.append(lab)
    return out_b, out_l


def _coco_name_to_thesis_idx(name: str) -> int | None:
    """Имя категории из COCO JSON → индекс 0..3 (как в обучении R-CNN)."""
    s = str(name).strip().lower().replace("_", " ")
    if s in ("drone", "uav", "quadcopter", "multirotor"):
        return 0
    if s in ("plane", "airplane", "aeroplane", "aircraft", "fixed wing", "fixed-wing"):
        return 1
    if s in ("helicopter", "heli"):
        return 2
    if s in ("bird",):
        return 3
    return None


def _coco_id_to_model_label(coco: dict) -> dict[int, int]:
    """Roboflow category_id (часто 1,2,…) → label torchvision 1..4."""
    out: dict[int, int] = {}
    for c in coco.get("categories", []):
        try:
            raw_id = int(c["id"])
        except (TypeError, ValueError, KeyError):
            continue
        ti = _coco_name_to_thesis_idx(str(c.get("name", "")))
        if ti is not None:
            out[raw_id] = ti + 1
    return out


def _thesis_idx_to_coco_cat_id(coco_gt: COCO) -> dict[int, int]:
    """thesis 0..3 → category_id как в JSON (для loadRes)."""
    out: dict[int, int] = {}
    for c in coco_gt.loadCats(coco_gt.getCatIds()):
        ti = _coco_name_to_thesis_idx(str(c.get("name", "")))
        if ti is not None:
            out[ti] = int(c["id"])
    return out


class DroneCocoDataset(Dataset):
    def __init__(self, img_dir: Path, ann_file: Path, aug_pipeline=None) -> None:
        self.img_dir = img_dir
        self.aug_pipeline = aug_pipeline
        with open(ann_file, encoding="utf-8") as f:
            coco = json.load(f)
        self.coco_id_to_label = _coco_id_to_model_label(coco)
        self.images = {int(im["id"]): im for im in coco["images"]}
        self.img_ids = list(self.images.keys())
        self.anns: dict[int, list] = {iid: [] for iid in self.img_ids}
        for ann in coco.get("annotations", []):
            try:
                iid = int(ann["image_id"])
            except (KeyError, TypeError, ValueError):
                continue
            if iid not in self.anns:
                continue
            self.anns[iid].append(ann)

    def __len__(self) -> int:
        return len(self.img_ids)

    def __getitem__(self, idx: int):
        img_id = self.img_ids[idx]
        img_info = self.images[img_id]
        img_path = self.img_dir / img_info["file_name"]
        if not img_path.is_file():
            img_path = self.img_dir / Path(img_info["file_name"]).name
        img = Image.open(img_path).convert("RGB")
        boxes, labels = [], []
        for ann in self.anns[img_id]:
            parsed = _parse_coco_bbox(ann)
            if parsed is None:
                continue
            x, y, w, h = parsed
            try:
                cid = int(ann["category_id"])
            except (TypeError, ValueError, KeyError):
                continue
            mlab = self.coco_id_to_label.get(cid)
            if mlab is None:
                continue
            boxes.append([x, y, x + w, y + h])
            labels.append(mlab)
        img_np = np.array(img)
        ih, iw = img_np.shape[0], img_np.shape[1]
        boxes, labels = _clip_xyxy_to_image(boxes, labels, ih, iw)
        if self.aug_pipeline and boxes:
            augmented = self.aug_pipeline(image=img_np, bboxes=boxes, labels=labels)
            img_np = augmented["image"]
            boxes = augmented["bboxes"]
            labels = augmented["labels"]
            ih2, iw2 = img_np.shape[0], img_np.shape[1]
            boxes, labels = _clip_xyxy_to_image(boxes, labels, ih2, iw2)
        if not boxes:
            boxes_t = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes_t = torch.tensor(boxes, dtype=torch.float32)
            labels_t = torch.tensor(labels, dtype=torch.int64)
        target = {
            "boxes": boxes_t,
            "labels": labels_t,
            "image_id": torch.tensor([img_id]),
            "area": (boxes_t[:, 3] - boxes_t[:, 1]) * (boxes_t[:, 2] - boxes_t[:, 0])
            if len(boxes_t)
            else torch.zeros(0),
            "iscrowd": torch.zeros(len(labels_t), dtype=torch.int64),
        }
        img_tensor = tvf.ToTensor()(Image.fromarray(img_np))
        return img_tensor, target


def collate_fn_rcnn(batch):
    return tuple(zip(*batch))


def _per_class_ap_from_eval(coco_eval: COCOeval):
    precision = coco_eval.eval["precision"]
    z = [0.0] * len(coco_eval.params.catIds)
    if precision is None:
        return z, z
    T, _, K, _, _ = precision.shape
    area_idx, maxdet_idx = 0, 2
    ap50, ap5095 = [], []
    for k in range(K):
        s0 = precision[0, :, k, area_idx, maxdet_idx]
        s0 = s0[s0 > -1]
        ap50.append(float(np.mean(s0)) if len(s0) else 0.0)
        tiers = []
        for ti in range(T):
            s = precision[ti, :, k, area_idx, maxdet_idx]
            s = s[s > -1]
            tiers.append(float(np.mean(s)) if len(s) else 0.0)
        ap5095.append(float(np.mean(tiers)))
    return ap50, ap5095


def _align_ap_to_thesis_names(coco_gt: COCO, cat_ids: list[int], vals: list[float], n: int = 4):
    """AP по порядку catIds → слоты DRONE/AIRPLANE/… по имени категории в GT, не по сырому id."""
    out = [0.0] * n
    for cid, v in zip(cat_ids, vals):
        cs = coco_gt.loadCats([int(cid)])
        if not cs:
            continue
        ti = _coco_name_to_thesis_idx(str(cs[0].get("name", "")))
        if ti is not None and 0 <= ti < n:
            out[ti] = float(v)
    return out


def evaluate_coco(model, test_ds: DroneCocoDataset, ann_file: Path, device: torch.device, conf_thresh: float = 0.5):
    model.eval()
    coco_gt = COCO(str(ann_file))
    _tid2coco = _thesis_idx_to_coco_cat_id(coco_gt)
    results = []
    loader = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=4, collate_fn=collate_fn_rcnn, pin_memory=True)
    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Night COCO eval"):
            images = [img.to(device) for img in images]
            preds = model(images)
            for pred, target in zip(preds, targets):
                img_id = target["image_id"].item()
                for box, score, label in zip(pred["boxes"], pred["scores"], pred["labels"]):
                    if score < conf_thresh:
                        continue
                    tidx = int(label.item()) - 1
                    coco_cat = _tid2coco.get(tidx)
                    if coco_cat is None:
                        continue
                    x1, y1, x2, y2 = box.cpu().tolist()
                    results.append(
                        {
                            "image_id": img_id,
                            "category_id": int(coco_cat),
                            "bbox": [x1, y1, x2 - x1, y2 - y1],
                            "score": score.item(),
                        }
                    )
    z4 = [0.0, 0.0, 0.0, 0.0]
    if not results:
        return {"mAP50": 0.0, "mAP50_95": 0.0, "per_class_mAP50": z4.copy(), "per_class_mAP50_95": z4.copy(), "recall": 0.0}
    coco_dt = coco_gt.loadRes(results)
    coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    ap50_raw, ap5095_raw = _per_class_ap_from_eval(coco_eval)
    cat_ids = list(coco_eval.params.catIds)
    per50 = _align_ap_to_thesis_names(coco_gt, cat_ids, ap50_raw)
    per5095 = _align_ap_to_thesis_names(coco_gt, cat_ids, ap5095_raw)
    return {
        "mAP50_95": round(float(coco_eval.stats[0]), 4),
        "mAP50": round(float(coco_eval.stats[1]), 4),
        "per_class_mAP50": [round(x, 4) for x in per50],
        "per_class_mAP50_95": [round(x, 4) for x in per5095],
        "recall": round(float(coco_eval.stats[8]), 4),
    }


_ANN_NAMES = (
    "instances_test.json",
    "instances_val.json",
    "instances_train.json",
    "_annotations.coco.json",
)


def _find_night_coco_ann(root: Path) -> Path | None:
    """Roboflow COCO: часто только instances_train.json или _annotations.coco.json в train/."""
    found: list[Path] = []
    for name in _ANN_NAMES:
        found.extend(root.rglob(name))
    if not found:
        return None
    order = {n: i for i, n in enumerate(_ANN_NAMES)}
    found.sort(key=lambda p: (order.get(p.name, 99), len(p.parts)))
    return found[0]


def _resolve_coco_image_dir(ann_path: Path) -> Path:
    """Папка с изображениями по первому file_name в COCO (train/images, рядом с json и т.д.)."""
    with open(ann_path, encoding="utf-8") as f:
        coco = json.load(f)
    imgs = coco.get("images") or []
    parent = ann_path.parent
    if imgs:
        fn = str(imgs[0].get("file_name", ""))
        base = Path(fn).name
        for cand in (
            parent / fn,
            parent / base,
            parent / "images" / base,
            parent / "images" / Path(fn).name,
        ):
            if cand.is_file():
                return cand.parent
        hits = [p for p in parent.rglob(base) if p.is_file()]
        if hits:
            return hits[0].parent
    for sub in ("images", "train", "valid", "test", "val"):
        d = parent / sub
        if d.is_dir() and (any(d.glob("*.jpg")) or any(d.glob("*.png")) or any(d.glob("*.jpeg"))):
            return d
    if parent.is_dir() and (any(parent.glob("*.jpg")) or any(parent.glob("*.png"))):
        return parent
    return parent / "images" if (parent / "images").is_dir() else parent


night_rcnn_metrics = {}
night_compare_rows = []

_ann = _find_night_coco_ann(NIGHT_COCO_DIR)
if _ann is None:
    print(
        f"Нет COCO JSON в {NIGHT_COCO_DIR} (ожидаются {_ANN_NAMES}). "
        "Проверьте распаковку night_drones_vs_planes.coco.zip (CELL 3)."
    )
elif not RCNN_WEIGHTS.exists():
    print(f"Нет весов: {RCNN_WEIGHTS}")
else:
    ANN_PATH = _ann
    COCO_IMG_DIR = _resolve_coco_image_dir(ANN_PATH)
    print("COCO images:", COCO_IMG_DIR, "| ann:", ANN_PATH)

    test_ds_rcnn = DroneCocoDataset(img_dir=COCO_IMG_DIR, ann_file=ANN_PATH, aug_pipeline=None)

    model_rcnn = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
    in_f = model_rcnn.roi_heads.box_predictor.cls_score.in_features
    model_rcnn.roi_heads.box_predictor = FastRCNNPredictor(in_f, NUM_CLASSES_RCNN)
    try:
        ckpt = torch.load(RCNN_WEIGHTS, map_location=DEVICE_RCNN, weights_only=False)
    except TypeError:
        ckpt = torch.load(RCNN_WEIGHTS, map_location=DEVICE_RCNN)
    _sd = ckpt.get("model_state_dict") or ckpt.get("state_dict") or ckpt
    model_rcnn.load_state_dict(_sd, strict=True)
    model_rcnn.to(DEVICE_RCNN)
    model_rcnn.eval()

    night_rcnn_metrics = evaluate_coco(model_rcnn, test_ds_rcnn, ANN_PATH, DEVICE_RCNN)
    night_rcnn_metrics["fps"] = None
    night_rcnn_metrics["size_mb"] = round(RCNN_WEIGHTS.stat().st_size / 1e6, 1)
    night_rcnn_metrics["class_names"] = CLASS_NAMES
    night_rcnn_metrics["dataset"] = "night_drones_vs_planes_coco"

    _test_jpg = sorted(COCO_IMG_DIR.rglob("*.jpg"), key=lambda p: p.name)[:50]
    if _test_jpg:
        _warm = [tvf.ToTensor()(Image.open(p).convert("RGB")).to(DEVICE_RCNN) for p in _test_jpg[:4]]
        with torch.no_grad():
            for _ in range(5):
                model_rcnn(_warm)
        t0 = time.perf_counter()
        Nf = min(50, len(_test_jpg))
        with torch.no_grad():
            for p in _test_jpg[:Nf]:
                model_rcnn([tvf.ToTensor()(Image.open(p).convert("RGB")).to(DEVICE_RCNN)])
        night_rcnn_metrics["fps"] = round(Nf / (time.perf_counter() - t0), 1)

    with open(RESULTS_DIR / "night_rcnn_metrics.json", "w", encoding="utf-8") as f:
        json.dump(night_rcnn_metrics, f, indent=2, ensure_ascii=False)
    print("night_rcnn_metrics:", night_rcnn_metrics)

# ── Сравнение YOLOv12s vs Faster R-CNN (ночной тест) ─────────────────────────────
import pandas as pd

_y12 = WEIGHTS_DIR / YOLO12S_NAME
_rows = []
_metrics_json = RESULTS_DIR / "night_yolo_metrics.json"
if _y12.exists() and _metrics_json.is_file():
    with open(_metrics_json, encoding="utf-8") as f:
        _ny = json.load(f)
    if "yolo12s" in _ny and isinstance(_ny["yolo12s"], dict):
        d = _ny["yolo12s"]
        _m50 = d.get("mAP50", d.get("map50"))
        _m95 = d.get("mAP50_95", d.get("map"))
        print(f"Сравнение: YOLO из {_metrics_json.name} → mAP50={_m50}, mAP50-95={_m95}")
        _rows.append(
            {
                "Model": "YOLOv12s",
                "mAP@0.5": float(_m50) if _m50 is not None else None,
                "mAP@0.5:0.95": float(_m95) if _m95 is not None else None,
                "Precision": float(d["precision"]) if d.get("precision") is not None else None,
                "Recall": float(d["recall"]) if d.get("recall") is not None else None,
                "FPS (T4)": d.get("fps"),
                "Size (MB)": d.get("size_mb"),
            }
        )
if night_rcnn_metrics:
    _rows.append(
        {
            "Model": "Faster R-CNN",
            "mAP@0.5": night_rcnn_metrics.get("mAP50"),
            "mAP@0.5:0.95": night_rcnn_metrics.get("mAP50_95"),
            "Precision": "N/A",
            "Recall": night_rcnn_metrics.get("recall"),
            "FPS (T4)": night_rcnn_metrics.get("fps"),
            "Size (MB)": night_rcnn_metrics.get("size_mb"),
        }
    )

if len(_rows) >= 1:
    df_cmp = pd.DataFrame(_rows).set_index("Model")
    print("\n" + "=" * 60)
    print("NIGHT — YOLOv12s vs Faster R-CNN (готовые веса)")
    print("=" * 60)
    print(df_cmp.to_string())
    df_cmp.to_csv(RESULTS_DIR / "night_compare_yolo12s_rcnn.csv")
    print(f"\nSaved → {RESULTS_DIR / 'night_compare_yolo12s_rcnn.csv'}")
else:
    print("Нет данных для сравнения: выполните CELL 6 (YOLO) и/или проверьте COCO + faster_rcnn_best.pth.")

## 2. Сводная таблица и графики (как в 04)


In [ ]:
# ── CELL 7: Таблица + CSV (без raise при пустых метриках — обновите файл в Colab) ─
import json
from pathlib import Path

import pandas as pd

# После перезапуска ядра может не быть CELL 5 — подставляем те же пути
try:
    RESULTS_DIR
except NameError:
    RESULTS_DIR = Path("/content/drive/MyDrive/Colab Notebooks/results/night_eval")
try:
    YOLO_WEIGHT_FILES
    _JSON_KEY
except NameError:
    YOLO_WEIGHT_FILES = [
        ("YOLOv12n", "yolo12n_drone_best.pt"),
        ("YOLOv12s", "yolo12s_drone_best.pt"),
        ("YOLO26n", "yolo26n_drone_best.pt"),
        ("YOLO26s", "yolo26s_drone_best.pt"),
    ]
    _JSON_KEY = {"YOLOv12n": "yolo12n", "YOLOv12s": "yolo12s", "YOLO26n": "yolo26n", "YOLO26s": "yolo26s"}

_KEY_TO_LABEL = {v: k for k, v in _JSON_KEY.items()}
_json_file = RESULTS_DIR / "night_yolo_metrics.json"
if not _json_file.is_file():
    raise FileNotFoundError(f"Нет файла: {_json_file}. Выполните CELL 6 или проверьте RESULTS_DIR.")

with open(_json_file, encoding="utf-8") as f:
    ny = json.load(f)


def _metrics_dict_to_row(model_label: str, d: object) -> dict | None:
    if not isinstance(d, dict):
        return None
    m50 = d.get("mAP50", d.get("map50"))
    if m50 is None:
        return None
    m95 = d.get("mAP50_95", d.get("map", d.get("mAP50-95")))
    if m95 is None:
        m95 = "N/A"
    return {
        "Model": model_label,
        "mAP@0.5": m50,
        "mAP@0.5:0.95": m95,
        "Precision": d.get("precision", "N/A"),
        "Recall": d.get("recall", "N/A"),
        "Size (MB)": d.get("size_mb", "N/A"),
    }


rows = []
for label, key in YOLO_WEIGHT_FILES:
    if key not in ny or not isinstance(ny[key], dict):
        continue
    r = _metrics_dict_to_row(label, ny[key])
    if r:
        rows.append(r)

if not rows:
    for _k, _v in ny.items():
        if _k in ("class_names", "_meta") or _k.startswith("_"):
            continue
        r = _metrics_dict_to_row(_KEY_TO_LABEL.get(_k, str(_k)), _v)
        if r:
            rows.append(r)

_skip = {"class_names", "_meta"}
_data_keys = [k for k in ny if k not in _skip and not str(k).startswith("_")]
print(f"Читаем: {_json_file}")
print(f"Верхнеуровневые ключи (данные): {_data_keys}")
if isinstance(ny.get("_meta"), dict):
    print("_meta['ok_models']:", ny["_meta"].get("ok_models"))

if not rows:
    print(
        "Нет блоков с mAP50 в JSON — файл записан, но CELL 6 не добавил метрики моделей "
        "(часто ok_models пустой: все val() упали или нет .pt в WEIGHTS_DIR)."
    )
    _m = ny.get("_meta") if isinstance(ny, dict) else {}
    if isinstance(_m, dict):
        if _m.get("val_errors"):
            print("Ошибки val() (см. CELL 6):", _m["val_errors"])
        if _m.get("skipped_weights"):
            print("Нет файлов весов:", _m["skipped_weights"])
    print("Исправьте data.yaml / пути (CELL 4), положите .pt в WEIGHTS_DIR и перезапустите CELL 6.")
    df_summary = pd.DataFrame(
        columns=["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "Size (MB)"]
    )
else:
    df_summary = pd.DataFrame(rows).set_index("Model")
    print("NIGHT — MODEL COMPARISON")
    print(df_summary.to_string())
    df_summary.to_csv(RESULTS_DIR / "night_model_comparison.csv")
    print(f"Saved → {RESULTS_DIR / 'night_model_comparison.csv'}")


In [ ]:
# ── CELL 8: Столбчатые диаграммы mAP ─────────────────────────────────────────
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

_csv = RESULTS_DIR / "night_model_comparison.csv"
if not _csv.exists():
    print("Нет night_model_comparison.csv — сначала успешно выполните CELL 6 и CELL 7.")
else:
    df_summary = pd.read_csv(_csv, index_col="Model")
    models = list(df_summary.index)
    if not models:
        print("Пропуск графика mAP: CSV без строк моделей.")
    else:
        colors = [MODEL_COLORS.get(m, "#7f8c8d") for m in models]
        map50_vals = [float(df_summary.loc[m, "mAP@0.5"]) for m in models]
        map5095_vals = [float(df_summary.loc[m, "mAP@0.5:0.95"]) for m in models]
        fig, ax = plt.subplots(figsize=(max(10, 1.5 + len(models)), 5))
        x = np.arange(len(models))
        w = 0.35
        ax.bar(x - w / 2, map50_vals, w, label="mAP@0.5", color=colors, alpha=0.9)
        ax.bar(x + w / 2, map5095_vals, w, label="mAP@0.5:0.95", color=colors, alpha=0.45)
        ax.set_xticks(x)
        ax.set_xticklabels(models, rotation=15, ha="right")
        ax.set_ylabel("mAP")
        ax.set_ylim(0, 1.0)
        ax.set_title("Night dataset — mAP (YOLO)")
        ax.legend()
        ax.grid(True, axis="y", alpha=0.3)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / "night_model_comparison.png", dpi=150)
        plt.show()


In [ ]:
# ── CELL 9: Per-class mAP@0.5 ─────────────────────────────────────────────────
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

with open(RESULTS_DIR / "night_yolo_metrics.json", encoding="utf-8") as f:
    _ny = json.load(f)
night_per_class = {}
for label, key in YOLO_WEIGHT_FILES:
    d = _ny.get(key)
    if isinstance(d, dict) and isinstance(d.get("per_class_mAP50"), list):
        night_per_class[label] = dict(zip(CLASS_NAMES, d["per_class_mAP50"]))

if not night_per_class:
    print("Нет per_class в night_yolo_metrics.json — перезапустите CELL 6.")
else:
    fig, ax = plt.subplots(figsize=(max(11, 2 + len(night_per_class) * 2), 5))
    n_classes = len(CLASS_NAMES)
    x = np.arange(n_classes)
    labels_order = list(night_per_class.keys())
    n_m = len(labels_order)
    width = min(0.8 / max(n_m, 1), 0.22)
    offsets = [width * (i - (n_m - 1) / 2) for i in range(n_m)]
    for off, lab in zip(offsets, labels_order):
        ap = [night_per_class[lab].get(c, 0) for c in CLASS_NAMES]
        ax.bar(x + off, ap, width, label=lab, color=MODEL_COLORS.get(lab, "#95a5a6"), alpha=0.88)
    ax.set_title("Night — Per-Class mAP@0.5", fontsize=13, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_NAMES, fontsize=11)
    ax.set_ylabel("AP@0.5")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "night_per_class_map.png", dpi=150)
    plt.show()


## 3. Confusion matrix и PR (YOLO)


In [ ]:
# ── CELL 10: Confusion matrices ───────────────────────────────────────────────
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from ultralytics import YOLO

def get_yolo_confusion(weights_path: Path, data_yaml: Path, conf: float = 0.25, iou: float = 0.5) -> np.ndarray:
    model = YOLO(str(weights_path))
    results = model.val(data=str(data_yaml), split=SPLIT_NIGHT, device=0, conf=conf, iou=iou, verbose=False, plots=False)
    cm = results.confusion_matrix.matrix
    row_sums = cm.sum(axis=1, keepdims=True).clip(min=1)
    return cm / row_sums


_yolo_for_cm = [(lab, WEIGHTS_DIR / fn) for lab, fn in YOLO_WEIGHT_FILES if (WEIGHTS_DIR / fn).exists()]
if not _yolo_for_cm or not torch.cuda.is_available():
    print("Пропуск confusion: нет весов или CUDA.")
else:
    n_cm = len(_yolo_for_cm)
    ncols = 2 if n_cm > 2 else n_cm
    nrows = int(np.ceil(n_cm / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4.5 * nrows))
    axes = np.atleast_1d(axes).ravel()
    fig.suptitle("Night — Confusion Matrices (normalized)", fontsize=13, fontweight="bold")
    tick_labels = list(CLASS_NAMES) + ["Background"]
    for ax, (title, wpath) in zip(axes, _yolo_for_cm):
        cm = get_yolo_confusion(wpath, YAML_PATH)
        n = min(cm.shape[0], len(tick_labels))
        sns.heatmap(
            cm[:n, :n],
            annot=True,
            fmt=".2f",
            cmap="Blues",
            ax=ax,
            xticklabels=tick_labels[:n],
            yticklabels=tick_labels[:n],
            vmin=0,
            vmax=1,
            linewidths=0.5,
        )
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
    for j in range(len(_yolo_for_cm), len(axes)):
        axes[j].set_visible(False)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "night_confusion_matrices.png", dpi=150)
    plt.show()


In [ ]:
# ── CELL 11: PR-кривые ─────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

_yolo_pr = [(lab, WEIGHTS_DIR / fn) for lab, fn in YOLO_WEIGHT_FILES if (WEIGHTS_DIR / fn).exists()]
if not _yolo_pr:
    print("Нет весов для PR.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.ravel()
    for ax, (model_name, wpath) in zip(axes, _yolo_pr):
        model = YOLO(str(wpath))
        results = model.val(data=str(YAML_PATH), split=SPLIT_NIGHT, device=0, verbose=False, plots=False)
        pr_data = {"precision": results.box.p, "recall": results.box.r, "ap50": results.box.ap50}
        for cls_idx, cls_name in enumerate(CLASS_NAMES):
            if cls_idx >= len(pr_data["recall"]):
                break
            ra = np.array(pr_data["recall"][cls_idx]) if hasattr(pr_data["recall"][cls_idx], "__len__") else [pr_data["recall"][cls_idx]]
            pa = np.array(pr_data["precision"][cls_idx]) if hasattr(pr_data["precision"][cls_idx], "__len__") else [pr_data["precision"][cls_idx]]
            ap = pr_data["ap50"][cls_idx] if cls_idx < len(pr_data["ap50"]) else 0
            c = PALETTE[cls_idx % len(PALETTE)]
            ax.plot(ra, pa, color=c, linewidth=2, label=f"{cls_name} (AP={ap:.3f})")
        ax.set_title(model_name)
        ax.set_xlabel("Recall")
        ax.set_ylabel("Precision")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
    for j in range(len(_yolo_pr), len(axes)):
        axes[j].set_visible(False)
    plt.suptitle("Night — Precision-Recall", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "night_pr_curves.png", dpi=150)
    plt.show()


## 4. COCO (ночной): путь для внешней оценки Faster R-CNN


In [ ]:
# ── CELL 12: Где лежит COCO после распаковки (для ручного запуска 03 / pycocotools)
from pathlib import Path

_ann = list(NIGHT_COCO_DIR.rglob("instances_*.json"))
print("Найденные COCO annotations:", _ann[:5])
if _ann:
    print("Пример: используйте эту папку как COCO_DIR в отдельном eval Faster R-CNN при совпадении category_id с обучением.")
else:
    print("В coco_unpack нет instances_*.json — проверьте структуру zip.")


In [ ]:
# ── CELL 13: Итог ─────────────────────────────────────────────────────────────
print("=" * 65)
print(f"NIGHT EVAL — артефакты: {RESULTS_DIR}")
print("=" * 65)
for f in [
    "night_yolo_metrics.json",
    "night_model_comparison.csv",
    "night_model_comparison.png",
    "night_per_class_map.png",
    "night_confusion_matrices.png",
    "night_pr_curves.png",
]:
    p = RESULTS_DIR / f
    print(f"  [{'OK' if p.exists() else '--'}] {f}")
